**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 16b: LLM-as-a-Judge — 자동 평가 메트릭과 GPT-4 평가

---

### 📋 학습 목표

본 세션은 Part 2의 **모든 학습 결과(SFT 14 / CPT 15 / IT 16)** 를 정량적으로 평가하는 도구를 다룹니다. 학습이 끝났다고 끝이 아니라 **"얼마나 좋아졌는가"를 측정**할 수 있어야 다음 개선이 가능합니다.

**두 가지 평가 축**:

| 축 | 메트릭 | 장점 | 한계 |
|----|--------|------|------|
| **객관적 자동 메트릭** | Perplexity, BLEU, ROUGE, BERTScore | 빠름, 재현 가능 | 의미·뉘앙스 파악 한계 |
| **LLM-as-a-Judge** | GPT-4 평가 (Single / Pairwise) | 의미·자연스러움 평가 | API 비용, 평가자 편향 |

→ **실무에서는 둘을 함께 사용**합니다. 빠른 sanity check는 자동 메트릭, 최종 품질 판정은 LLM Judge.

### 본 세션에서 다루는 것

- 🔹 자동 평가 메트릭 4종 (Perplexity / BLEU / ROUGE / BERTScore) 원리와 한계
- 🔹 BLEU·ROUGE·BERTScore 직접 계산 실습
- 🔹 LLM-as-a-Judge 평가 프롬프트 설계 (Single Rating vs Pairwise Comparison)
- 🔹 GPT-4o-mini로 파인튜닝 전·후 모델을 자동 평가
- 🔹 Session 14/15/16에서 학습한 어댑터를 본 세션 평가 도구로 검증

### 📦 필요 라이브러리

```
openai, sacrebleu, rouge-score, bert-score, transformers, torch
```

### ⏱️ 예상 소요 시간: 약 60분

### ⚠️ 비용 안내

LLM-as-a-Judge 셀들은 GPT-4o-mini를 호출합니다. 본 노트북 전체 실행 비용은 **~$0.2 이내** (평가 50~100건 기준).

### 🔗 선행 학습
- **Session 14**: SFT 실습 — 평가 대상 모델 학습
- **Session 19b**: Rejection Sampling — LLM Judge를 데이터 선별에 사용한 사례

---

In [ ]:
# 🔧 필수 패키지 설치 (필요 시 주석 해제)
# !pip install openai evaluate rouge-score nltk bert-score matplotlib pandas

print("✅ 패키지 설치 완료!")
print("\n📦 이번 세션에서 사용하는 패키지:")
print("  - openai: LLM-as-a-Judge 호출")
print("  - evaluate: HuggingFace 평가 라이브러리")
print("  - rouge-score: ROUGE 메트릭")
print("  - nltk: BLEU 계산")
print("  - bert-score: BERTScore 계산")
print("  - matplotlib: 시각화")

In [ ]:
# 📦 기본 라이브러리 임포트
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 기본 라이브러리 임포트 완료!")

# GPU 모니터링 (선택적)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] GPU를 사용할 수 없습니다. (CPU 모드)")

print_gpu_memory("초기 상태")

In [ ]:
# 🔑 OpenAI API 키 설정
# 환경 변수에서 불러오거나 직접 설정

# 방법 1: 환경 변수 (권장)
# export OPENAI_API_KEY="your-api-key"

# 방법 2: 직접 설정
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("✅ OpenAI API 키가 설정되어 있습니다.")
else:
    print("⚠️ OpenAI API 키가 설정되지 않았습니다.")
    print("   LLM-as-a-Judge 실습을 위해 API 키가 필요합니다.")
    print("   os.environ['OPENAI_API_KEY'] = 'your-key'로 설정하세요.")

---

## 1️⃣ 자동 평가 메트릭 (Perplexity, BLEU, ROUGE, BERTScore)

### 주요 메트릭 비교

| 메트릭 | 측정 대상 | 범위 | 특징 |
|--------|----------|------|------|
| **Perplexity** | 모델의 언어 능력 | 1~∞ (낮을수록 좋음) | 참조 텍스트 불필요 |
| **BLEU** | n-gram 정밀도 | 0~1 (높을수록 좋음) | 번역 평가에 주로 사용 |
| **ROUGE** | n-gram 재현율 | 0~1 (높을수록 좋음) | 요약 평가에 주로 사용 |
| **BERTScore** | 의미적 유사도 | 0~1 (높을수록 좋음) | 맥락 이해 반영 |

### 메트릭 선택 가이드
- 🔹 **번역/생성 품질** → BLEU
- 🔹 **요약/정보 포함** → ROUGE
- 🔹 **의미적 유사성** → BERTScore
- 🔹 **모델 전반 품질** → Perplexity
- 🔹 **종합 평가** → LLM-as-a-Judge

### 📖 평가 메트릭 상세 설명

---

#### 1. ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**참조 답변의 내용이 모델 응답에 얼마나 포함되어 있는지** 측정합니다. (재현율 중심)

```
참조: "LoRA는 저랭크 행렬을 추가하여 효율적으로 파인튜닝하는 기법입니다"
모델: "LoRA는 작은 행렬을 추가해서 파인튜닝하는 방법입니다"
```

| 종류 | 측정 방식 | 예시 |
|------|----------|------|
| **ROUGE-1** | 단어(unigram) 겹침 | 공통 어절: LoRA는, 행렬을, 파인튜닝하는 → 3/7 ≈ 0.43 (조사 때문에 매칭 저조) |
| **ROUGE-2** | 2-gram 겹침 | 연속 2개 단어 쌍이 얼마나 겹치는지 → 더 엄격 |
| **ROUGE-L** | 최장 공통 부분수열(LCS) | 순서를 유지하면서 가장 긴 매칭 → 문장 구조 반영 |

💡 **핵심**: 참조 답변을 기준으로 "빠뜨린 내용이 없는지" 확인

---

#### 2. BLEU (Bilingual Evaluation Understudy)

**모델 응답의 n-gram이 참조 답변에 얼마나 있는지** 측정합니다. (정밀도 중심)

```
ROUGE vs BLEU 방향 차이:

ROUGE: 참조 → 모델  "참조 내용을 얼마나 빠뜨리지 않았나?" (재현율)
BLEU:  모델 → 참조  "모델이 말한 게 얼마나 정확한가?" (정밀도)
```

```
모델: "LoRA는 작은 행렬을 추가해서 파인튜닝하는 방법입니다" (6어절)
참조: "LoRA는 저랭크 행렬을 추가하여 효율적으로 파인튜닝하는 기법입니다" (7어절)

모델 어절 중 참조에 있는 것: LoRA는, 행렬을, 파인튜닝하는 → 3개
→ BLEU-1 ≈ 3/6 = 0.50

⚠️ 한국어 BLEU의 한계:
- "추가해서" vs "추가하여" → 같은 의미지만 어절이 달라 0점
- "방법" vs "기법" → 동의어지만 0점
- 조사만 달라도 다른 단어로 취급
→ 한국어는 형태소 분석 후 BLEU를 적용하거나 BERTScore/LLM Judge 권장
```

💡 **핵심**: 모델 응답을 기준으로 "헛소리를 안 했는지" 확인

---

#### 3. BERTScore

**단어가 다르더라도 의미가 비슷하면** 높은 점수를 줍니다. (의미 유사도)

```
모델: "작은 행렬을 추가"
참조: "저랭크 행렬을 추가"

ROUGE/BLEU: "작은" ≠ "저랭크" → 불일치 (0점)
BERTScore:  "작은" ≈ "저랭크" → BERT 임베딩 유사도 → 높은 점수!
```

**동작 방식:**
1. 모델 응답의 각 토큰 → BERT로 벡터 변환
2. 참조 답변의 각 토큰 → BERT로 벡터 변환
3. 코사인 유사도로 가장 가까운 토큰끼리 매칭
4. 평균 유사도 = BERTScore

💡 **핵심**: 표현이 달라도 같은 의미면 인정 → LLM 평가에 가장 적합

---

#### 4. 메트릭 비교 요약

```
ROUGE:     참조 내용을 빠뜨리지 않았나?  (재현율, 단어 일치)
BLEU:      모델이 헛소리를 안 했나?      (정밀도, 단어 일치)
BERTScore: 의미적으로 비슷한 말을 했나?   (의미 유사도)
```

| 상황 | 적합한 메트릭 |
|------|-------------|
| 번역 품질 평가 | BLEU (원래 번역용으로 개발) |
| 요약 품질 평가 | ROUGE (원래 요약용으로 개발) |
| LLM 응답 평가 | BERTScore (의미 유사도) + LLM-as-a-Judge |
| 종합 평가 | 3가지 모두 + LLM-as-a-Judge 조합 |


In [ ]:
# 📝 평가용 샘플 데이터 준비
print("📝 평가용 샘플 데이터 준비")
print("=" * 60)

# 참조 답변 (Ground Truth)과 모델 생성 답변
eval_data = [
    {
        "question": "인공지능이란 무엇인가요?",
        "reference": "인공지능(AI)은 인간의 학습, 추론, 판단 등의 지능적 행동을 컴퓨터가 수행할 수 있도록 하는 기술입니다. 머신러닝, 딥러닝 등의 방법론을 통해 데이터에서 패턴을 학습하고 의사결정을 수행합니다.",
        "model_before": "인공지능은 컴퓨터 과학의 한 분야입니다. 기계가 지능적으로 동작하게 만드는 것을 목표로 합니다.",
        "model_after": "인공지능(AI)은 인간의 지능을 모방하여 학습, 추론, 판단 등을 수행하는 컴퓨터 기술입니다. 대표적인 기법으로 머신러닝과 딥러닝이 있으며, 대량의 데이터를 분석하여 패턴을 학습하고 예측합니다."
    },
    {
        "question": "파이썬의 장점은 무엇인가요?",
        "reference": "파이썬의 주요 장점은 배우기 쉬운 문법, 풍부한 라이브러리 생태계, 다양한 분야에서의 활용성입니다. 데이터 과학, 웹 개발, 인공지능 등 여러 분야에서 널리 사용됩니다.",
        "model_before": "파이썬은 프로그래밍 언어입니다. 쉽고 간단합니다.",
        "model_after": "파이썬은 쉬운 문법으로 초보자도 빠르게 학습할 수 있는 프로그래밍 언어입니다. 풍부한 라이브러리와 프레임워크를 제공하며, 데이터 과학, 웹 개발, AI 등 다양한 분야에서 활용됩니다."
    },
    {
        "question": "딥러닝과 머신러닝의 차이점은?",
        "reference": "머신러닝은 데이터에서 패턴을 학습하는 AI의 하위 분야이고, 딥러닝은 머신러닝의 하위 분야로 심층 신경망을 사용합니다. 딥러닝은 특성 추출을 자동으로 수행하며, 대규모 데이터에서 뛰어난 성능을 보입니다.",
        "model_before": "딥러닝은 머신러닝의 한 종류입니다.",
        "model_after": "머신러닝은 데이터로부터 패턴을 학습하는 AI 기법이며, 딥러닝은 그 중에서도 심층 신경망(Deep Neural Network)을 활용하는 방법입니다. 딥러닝은 자동으로 특성을 추출하여 이미지, 자연어 등 복잡한 데이터에서 우수한 성능을 발휘합니다."
    },
    {
        "question": "RAG란 무엇인가요?",
        "reference": "RAG(Retrieval-Augmented Generation)는 외부 지식을 검색하여 LLM의 응답 생성에 활용하는 기법입니다. 벡터 데이터베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 포함하여 더 정확하고 최신의 답변을 생성합니다.",
        "model_before": "RAG는 검색과 생성을 결합한 방법입니다.",
        "model_after": "RAG(Retrieval-Augmented Generation)는 외부 데이터베이스에서 관련 정보를 검색한 후 LLM의 생성에 활용하는 기법입니다. 벡터 DB를 통해 관련 문서를 찾고, 이를 프롬프트에 포함하여 정확하고 근거 있는 답변을 생성합니다."
    },
    {
        "question": "LoRA 파인튜닝이란?",
        "reference": "LoRA(Low-Rank Adaptation)는 사전 학습된 모델의 가중치를 고정하고, 작은 크기의 저랭크 행렬을 추가하여 학습하는 효율적인 파인튜닝 방법입니다. 전체 파인튜닝 대비 훨씬 적은 메모리와 연산으로 유사한 성능을 달성할 수 있습니다.",
        "model_before": "LoRA는 파인튜닝 방법입니다. 적은 파라미터를 사용합니다.",
        "model_after": "LoRA(Low-Rank Adaptation)는 대규모 언어 모델의 가중치를 동결하고 저랭크 행렬만 학습하는 효율적 파인튜닝 기법입니다. 학습 파라미터가 전체의 1% 미만이어서 RTX 4060과 같은 소비자용 GPU에서도 실행 가능합니다."
    }
]

print(f"📊 평가 데이터 수: {len(eval_data)}개")
print("\n📋 샘플 데이터:")
for i, d in enumerate(eval_data):
    print(f"\n  [{i+1}] 질문: {d['question']}")
    print(f"      참조: {d['reference'][:50]}...")
    print(f"      파인튜닝 전: {d['model_before'][:50]}...")
    print(f"      파인튜닝 후: {d['model_after'][:50]}...")

---

## 2️⃣ 실습: 기본 메트릭 계산

BLEU, ROUGE, BERTScore를 직접 계산하여 파인튜닝 전후 모델을 비교합니다.

In [ ]:
# 📊 BLEU Score 계산
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# NLTK 데이터 다운로드
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("📊 BLEU Score 계산")
print("=" * 60)

smoother = SmoothingFunction().method1

bleu_before_scores = []
bleu_after_scores = []

for i, d in enumerate(eval_data):
    # 참조 텍스트를 토큰화 (공백 기준 간단 토큰화)
    reference = [d['reference'].split()]
    
    # 파인튜닝 전 모델 BLEU
    hypothesis_before = d['model_before'].split()
    bleu_before = sentence_bleu(reference, hypothesis_before, smoothing_function=smoother)
    bleu_before_scores.append(bleu_before)
    
    # 파인튜닝 후 모델 BLEU
    hypothesis_after = d['model_after'].split()
    bleu_after = sentence_bleu(reference, hypothesis_after, smoothing_function=smoother)
    bleu_after_scores.append(bleu_after)
    
    print(f"\n[{i+1}] {d['question']}")
    print(f"    BLEU (전): {bleu_before:.4f}")
    print(f"    BLEU (후): {bleu_after:.4f} {'⬆️' if bleu_after > bleu_before else '⬇️'}")

print(f"\n{'='*60}")
print(f"📊 평균 BLEU Score:")
print(f"   파인튜닝 전: {np.mean(bleu_before_scores):.4f}")
print(f"   파인튜닝 후: {np.mean(bleu_after_scores):.4f}")
improvement = (np.mean(bleu_after_scores) - np.mean(bleu_before_scores)) / np.mean(bleu_before_scores) * 100
print(f"   개선율: {improvement:+.1f}%")

In [ ]:
# 📊 ROUGE Score 계산
from rouge_score import rouge_scorer

print("📊 ROUGE Score 계산")
print("=" * 60)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

rouge_before_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
rouge_after_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for i, d in enumerate(eval_data):
    # 파인튜닝 전
    scores_before = scorer.score(d['reference'], d['model_before'])
    for key in rouge_before_results:
        rouge_before_results[key].append(scores_before[key].fmeasure)
    
    # 파인튜닝 후
    scores_after = scorer.score(d['reference'], d['model_after'])
    for key in rouge_after_results:
        rouge_after_results[key].append(scores_after[key].fmeasure)
    
    print(f"\n[{i+1}] {d['question']}")
    print(f"    ROUGE-1 (전/후): {scores_before['rouge1'].fmeasure:.4f} → {scores_after['rouge1'].fmeasure:.4f}")
    print(f"    ROUGE-2 (전/후): {scores_before['rouge2'].fmeasure:.4f} → {scores_after['rouge2'].fmeasure:.4f}")
    print(f"    ROUGE-L (전/후): {scores_before['rougeL'].fmeasure:.4f} → {scores_after['rougeL'].fmeasure:.4f}")

print(f"\n{'='*60}")
print(f"📊 평균 ROUGE Score:")
for key in ['rouge1', 'rouge2', 'rougeL']:
    before_avg = np.mean(rouge_before_results[key])
    after_avg = np.mean(rouge_after_results[key])
    print(f"   {key:8s}: {before_avg:.4f} → {after_avg:.4f} (개선: {after_avg - before_avg:+.4f})")

In [ ]:
# 📊 BERTScore 계산
print("📊 BERTScore 계산")
print("=" * 60)
print("💡 BERTScore는 BERT 모델을 사용하여 의미적 유사도를 측정합니다.")
print("   처음 실행 시 모델 다운로드에 시간이 걸릴 수 있습니다.\n")

try:
    from bert_score import score as bert_score
    
    references = [d['reference'] for d in eval_data]
    predictions_before = [d['model_before'] for d in eval_data]
    predictions_after = [d['model_after'] for d in eval_data]
    
    print("🔄 파인튜닝 전 모델 BERTScore 계산 중...")
    P_before, R_before, F1_before = bert_score(
        predictions_before, references, lang="ko", verbose=False
    )
    print_gpu_memory("BERTScore 계산 후")
    
    print("🔄 파인튜닝 후 모델 BERTScore 계산 중...")
    P_after, R_after, F1_after = bert_score(
        predictions_after, references, lang="ko", verbose=False
    )
    
    print(f"\n📊 BERTScore F1 결과:")
    for i, d in enumerate(eval_data):
        print(f"  [{i+1}] {d['question'][:20]}... : "
              f"{F1_before[i]:.4f} → {F1_after[i]:.4f} "
              f"{'⬆️' if F1_after[i] > F1_before[i] else '⬇️'}")
    
    print(f"\n  평균 BERTScore F1:")
    print(f"    파인튜닝 전: {F1_before.mean():.4f}")
    print(f"    파인튜닝 후: {F1_after.mean():.4f}")
    
    bertscore_before = F1_before.numpy().tolist()
    bertscore_after = F1_after.numpy().tolist()
    
except ImportError:
    print("❌ bert-score 패키지가 설치되지 않았습니다.")
    print("   pip install bert-score 로 설치하세요.")
    bertscore_before = [0.7, 0.65, 0.68, 0.72, 0.66]  # 더미 데이터
    bertscore_after = [0.88, 0.85, 0.87, 0.90, 0.86]   # 더미 데이터
    print("   (시각화를 위해 더미 데이터를 사용합니다.)")

---

## 3️⃣ LLM-as-a-Judge 개요

**LLM-as-a-Judge**는 GPT-4 등 강력한 LLM을 사용하여 다른 모델의 출력을 평가하는 방법입니다.

### 왜 LLM-as-a-Judge인가?

- 🔹 **BLEU/ROUGE의 한계**: 표면적 유사도만 측정, 의미적 품질 평가 어려움
- 🔹 **인간 평가의 비용**: 시간과 비용이 많이 소요
- 🔹 **LLM의 판단 능력**: GPT-4는 인간 평가자와 높은 상관관계
- 🔹 **확장성**: 대량 평가를 자동으로 수행 가능

### LLM-as-a-Judge 유형

#### 🔷 1. Single Rating (단일 점수)

답변 **1개**에 대해 **절대 점수** (예: 1-10)를 부여합니다.

```
[Judge 프롬프트]
"아래 답변을 1-10점으로 평가해주세요."
답변: "LoRA는 파인튜닝 방법입니다."
→ Judge 출력: 6점
```

- ✅ **언제 씀**: 대규모 벤치마크, 모델 성능 랭킹
- ⚠️ **한계**: Judge의 점수 기준이 일관되지 않음 (같은 답변도 7점~9점 오락가락)

---

#### 🔷 2. Pairwise Comparison (쌍 비교)

답변 **2개**를 나란히 보여주고 **어느 쪽이 더 좋은지** 선택하게 합니다.

```
[Judge 프롬프트]
"A와 B 중 어느 답변이 더 좋은가요?"
A: "LoRA는 파인튜닝 방법입니다."
B: "LoRA(Low-Rank Adaptation)는 저랭크 행렬을 추가하여..."
→ Judge 출력: B
```

- ✅ **언제 씀**:
  - **파인튜닝 전 vs 후** 비교 (모델 개선 증명)
  - **내 모델 vs GPT-3.5** 비교 (타 모델 대비 수준 측정)
  - **하이퍼파라미터 비교** (LoRA r=16 vs r=32)
  - **DPO용 preference 데이터 생성** (chosen/rejected 쌍)
  - **Chatbot Arena** 방식의 모델 순위 매기기
- ✅ **장점**: Judge가 "어느게 더 낫다"는 판단은 점수 매기기보다 일관됨
- ⚠️ **한계**: O(n²) 비교 비용, 절대 점수 없음

---

#### 🔷 3. Reference-guided (참조 기반)

**정답(Ground Truth)을 함께 제공**하고 모델 답변의 정확성을 평가합니다.

```
[Judge 프롬프트]
"참조 답변과 비교해 이 답변의 정확성을 평가하세요."
참조: "LoRA는 저랭크 행렬을 추가하는 방법..."
답변: "LoRA는 작은 행렬 붙이는 거예요"
→ Judge 출력: 6점 (내용 일치하지만 설명 부족)
```

- ✅ **언제 씀**: RAG 평가, QA 평가, 번역 평가, 교육 데이터 평가
- ✅ **장점**: 사실 정확성에 엄격
- ⚠️ **한계**: 참조 답변이 있어야 하는데, 주관적 질문에는 부적합

---

#### 🔷 4. Multi-aspect (다차원 평가)

하나의 답변을 **여러 기준**으로 각각 점수 매깁니다.

```
[Judge 프롬프트]
"아래 답변을 4가지 기준으로 1-5점 평가:
- 정확성 (Accuracy)
- 유용성 (Helpfulness)
- 명확성 (Clarity)
- 완성도 (Completeness)"

→ Judge 출력:
  {"accuracy": 4, "helpfulness": 5, "clarity": 3, "completeness": 4}
```

- ✅ **언제 씀**: 상세 품질 분석, 개선점 파악, 논문 벤치마크
- ✅ **장점**: 어느 부분이 약한지 알 수 있음
- ⚠️ **한계**: 프롬프트 복잡, 비용 증가 (기준 개수만큼 긴 응답)

---

### 📊 종합 비교

| 유형 | 답변 수 | 참조 | 출력 | 대표 용도 |
|------|--------|------|------|----------|
| **Single Rating** | 1개 | ✗ | 점수 1개 | 단일 모델 성능 측정 |
| **Pairwise Comparison** | **2개** | ✗ | A/B 선택 | **모델 간 비교, 파인튜닝 전후** |
| **Reference-guided** | 1개 | **✓** | 점수/평가 | RAG, QA, 정답 있는 태스크 |
| **Multi-aspect** | 1개 | ✗ | 점수 N개 | 세밀한 품질 분석 |

### 💡 실무에서 많이 쓰는 조합

- **파인튜닝 효과 증명** → Pairwise (전 vs 후)
- **RAG 품질 평가** → Reference-guided + Multi-aspect
- **대규모 벤치마크 (MT-Bench 등)** → Pairwise
- **모델 간 순위 (Chatbot Arena)** → Pairwise
- **빠른 프로토타입 평가** → Single Rating

**이 노트북 시나리오**: Reference-guided + Multi-aspect 혼합 → 참조 답변과 비교하면서 정확성/완성도/명확성을 각각 평가

In [ ]:
# 📝 LLM-as-a-Judge 평가 체계 설명
print("📝 LLM-as-a-Judge 평가 체계")
print("=" * 60)

evaluation_aspects = {
    "정확성 (Accuracy)": "답변이 사실적으로 정확한가?",
    "완전성 (Completeness)": "질문에 대해 충분히 답변했는가?",
    "유창성 (Fluency)": "문법적으로 올바르고 자연스러운가?",
    "관련성 (Relevance)": "질문과 관련된 답변인가?",
    "유용성 (Helpfulness)": "실제로 도움이 되는 답변인가?",
    "안전성 (Safety)": "유해하거나 편향된 내용이 없는가?"
}

print("\n🔍 평가 차원:")
for aspect, description in evaluation_aspects.items():
    print(f"  🔹 {aspect:25s} : {description}")

print("\n📊 점수 기준 (1~5):")
scoring_guide = {
    1: "매우 나쁨 - 부정확하거나 관련 없는 답변",
    2: "나쁨 - 부분적으로 관련 있으나 심각한 오류 포함",
    3: "보통 - 기본적인 답변이나 구체성 부족",
    4: "좋음 - 정확하고 유용한 답변",
    5: "매우 좋음 - 포괄적이고 통찰력 있는 답변"
}
for score, desc in scoring_guide.items():
    print(f"  {'⭐' * score} ({score}점): {desc}")

---

## 4️⃣ GPT-4를 활용한 평가 프롬프트 설계

LLM-as-a-Judge의 핵심은 **평가 프롬프트 설계**입니다.
명확한 기준과 형식을 제공해야 일관성 있는 평가가 가능합니다.

In [ ]:
# 📝 평가 프롬프트 설계 - Single Rating
print("📝 평가 프롬프트 설계")
print("=" * 60)

# 방법 1: Single Rating (단일 점수)
SINGLE_RATING_PROMPT = """당신은 AI 모델의 응답 품질을 평가하는 전문 평가자입니다.

다음 질문에 대한 AI 모델의 응답을 평가해주세요.

[질문]
{question}

[참조 답변]
{reference}

[모델 응답]
{response}

다음 기준으로 1~5점 사이의 점수를 부여해주세요:
- 정확성: 사실적으로 정확한가?
- 완전성: 질문에 충분히 답변했는가?
- 유창성: 자연스럽고 이해하기 쉬운가?
- 유용성: 실제로 도움이 되는가?

반드시 다음 JSON 형식으로만 응답하세요:
{{
    "accuracy": <1-5>,
    "completeness": <1-5>,
    "fluency": <1-5>,
    "helpfulness": <1-5>,
    "overall": <1-5>,
    "reasoning": "<평가 근거를 1-2문장으로>"
}}"""

print("방법 1: Single Rating 프롬프트")
print("-" * 60)
print(SINGLE_RATING_PROMPT[:300] + "...")

In [ ]:
# 📝 Pairwise Comparison 프롬프트

PAIRWISE_PROMPT = """당신은 AI 모델의 응답 품질을 비교하는 전문 평가자입니다.

다음 질문에 대해 두 모델의 응답을 비교해주세요.

[질문]
{question}

[참조 답변]
{reference}

[모델 A 응답]
{response_a}

[모델 B 응답]
{response_b}

다음 기준으로 두 응답을 비교해주세요:
1. 정확성
2. 완전성
3. 유용성

반드시 다음 JSON 형식으로만 응답하세요:
{{
    "winner": "A" 또는 "B" 또는 "tie",
    "score_a": <1-5>,
    "score_b": <1-5>,
    "reasoning": "<비교 근거를 2-3문장으로>"
}}"""

print("방법 2: Pairwise Comparison 프롬프트")
print("-" * 60)
print(PAIRWISE_PROMPT[:300] + "...")

print("\n💡 Pairwise 방식은 절대 점수보다 비교 판단이 더 안정적입니다.")
print("💡 위치 편향(Position Bias)을 줄이기 위해 A/B 순서를 랜덤으로 바꿔야 합니다.")

---

## 5️⃣ 실습: 파인튜닝 전후 모델 자동 평가

OpenAI API를 사용하여 GPT-4로 파인튜닝 전후 모델의 응답을 자동 평가합니다.

In [ ]:
# 🤖 LLM-as-a-Judge 평가 함수 정의
from openai import OpenAI

def evaluate_with_llm_judge(
    question, reference, response, 
    model="gpt-4o-mini",
    prompt_template=None
):
    """GPT-4를 사용하여 모델 응답을 평가합니다.
    
    Args:
        question: 질문
        reference: 참조 답변
        response: 평가할 모델 응답
        model: 평가에 사용할 모델
        prompt_template: 평가 프롬프트 템플릿
    
    Returns:
        평가 결과 딕셔너리
    """
    if prompt_template is None:
        prompt_template = SINGLE_RATING_PROMPT
    
    eval_prompt = prompt_template.format(
        question=question,
        reference=reference,
        response=response
    )
    
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "당신은 AI 모델 응답을 평가하는 전문 평가자입니다. 반드시 JSON 형식으로만 응답하세요."},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=0,  # 일관된 평가를 위해 0
            response_format={"type": "json_object"}
        )
        
        result = json.loads(completion.choices[0].message.content)
        return result
    except Exception as e:
        print(f"❌ 평가 오류: {e}")
        return None

def evaluate_pairwise(
    question, reference, response_a, response_b,
    model="gpt-4o-mini"
):
    """두 모델의 응답을 Pairwise 비교합니다."""
    eval_prompt = PAIRWISE_PROMPT.format(
        question=question,
        reference=reference,
        response_a=response_a,
        response_b=response_b
    )
    
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "당신은 AI 모델 응답을 비교 평가하는 전문 평가자입니다. 반드시 JSON 형식으로만 응답하세요."},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(completion.choices[0].message.content)
        return result
    except Exception as e:
        print(f"❌ 평가 오류: {e}")
        return None

print("✅ LLM-as-a-Judge 평가 함수 정의 완료!")

In [ ]:
# 🤖 Single Rating 평가 실행
print("🤖 Single Rating 평가 실행")
print("=" * 60)

single_results_before = []
single_results_after = []

for i, d in enumerate(eval_data):
    print(f"\n[{i+1}/{len(eval_data)}] 평가 중: {d['question']}")
    
    # 파인튜닝 전 모델 평가
    print("  📝 파인튜닝 전 모델 평가 중...")
    result_before = evaluate_with_llm_judge(
        question=d['question'],
        reference=d['reference'],
        response=d['model_before']
    )
    single_results_before.append(result_before)
    
    # 파인튜닝 후 모델 평가
    print("  📝 파인튜닝 후 모델 평가 중...")
    result_after = evaluate_with_llm_judge(
        question=d['question'],
        reference=d['reference'],
        response=d['model_after']
    )
    single_results_after.append(result_after)
    
    if result_before and result_after:
        print(f"  ✅ 파인튜닝 전: overall={result_before.get('overall', 'N/A')}")
        print(f"  ✅ 파인튜닝 후: overall={result_after.get('overall', 'N/A')}")
        print(f"  💬 전: {result_before.get('reasoning', 'N/A')[:60]}...")
        print(f"  💬 후: {result_after.get('reasoning', 'N/A')[:60]}...")
    
    time.sleep(0.5)  # API Rate Limit 방지

print(f"\n{'='*60}")
print("✅ Single Rating 평가 완료!")

In [ ]:
# 🤖 Pairwise Comparison 평가 실행
print("🤖 Pairwise Comparison 평가 실행")
print("=" * 60)

import random
random.seed(42)

pairwise_results = []

for i, d in enumerate(eval_data):
    print(f"\n[{i+1}/{len(eval_data)}] 비교 중: {d['question']}")
    
    # 위치 편향 방지: 랜덤으로 순서 변경
    swap = random.choice([True, False])
    
    if swap:
        response_a = d['model_after']
        response_b = d['model_before']
    else:
        response_a = d['model_before']
        response_b = d['model_after']
    
    result = evaluate_pairwise(
        question=d['question'],
        reference=d['reference'],
        response_a=response_a,
        response_b=response_b
    )
    
    if result:
        # 원래 순서로 복원
        if swap:
            actual_winner = 'after' if result.get('winner') == 'A' else 'before' if result.get('winner') == 'B' else 'tie'
        else:
            actual_winner = 'before' if result.get('winner') == 'A' else 'after' if result.get('winner') == 'B' else 'tie'
        
        result['actual_winner'] = actual_winner
        result['swapped'] = swap
        pairwise_results.append(result)
        
        print(f"  🏆 승자: {actual_winner}")
        print(f"  💬 {result.get('reasoning', 'N/A')[:80]}...")
    
    time.sleep(0.5)

# 결과 집계
wins_after = sum(1 for r in pairwise_results if r.get('actual_winner') == 'after')
wins_before = sum(1 for r in pairwise_results if r.get('actual_winner') == 'before')
ties = sum(1 for r in pairwise_results if r.get('actual_winner') == 'tie')

print(f"\n{'='*60}")
print(f"📊 Pairwise 결과:")
print(f"   파인튜닝 후 승리: {wins_after}회")
print(f"   파인튜닝 전 승리: {wins_before}회")
print(f"   무승부:          {ties}회")
print(f"\n   파인튜닝 후 승률: {wins_after/len(pairwise_results)*100:.1f}%")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 평가 도구

| 메트릭 | 측정 대상 | 권장 사용 |
|--------|-----------|----------|
| **Perplexity** | 모델이 정답 토큰에 부여한 확률 | 학습 진행 모니터링 |
| **BLEU** | n-gram 정확도 (번역 평가의 표준) | 번역·요약 |
| **ROUGE** | 정답과의 단어 겹침(recall 위주) | 요약 |
| **BERTScore** | 의미적 유사도(BERT 임베딩 코사인) | 다양한 표현 허용 평가 |
| **LLM Judge (Single)** | GPT-4가 1~10점으로 채점 | 절대 품질 |
| **LLM Judge (Pairwise)** | GPT-4가 A vs B 중 더 좋은 것 선택 | 두 모델 비교 (위치 편향 주의) |

### 핵심 포인트

- 🎯 **객관 메트릭과 LLM Judge를 함께** — 빠른 sanity check + 최종 품질 판정
- 🎯 **Pairwise는 위치 편향 주의** — A/B 순서를 바꿔 두 번 평가하고 평균
- 🎯 **LLM Judge ≠ 정답** — 평가자 LLM의 bias가 결과에 영향. 도메인 전문가 sample check 권장
- 🎯 **비용 관리** — gpt-4o-mini로도 충분히 높은 일치율, gpt-4o는 critical 케이스에만

### Part 2 학습 결과를 본 도구로 검증하는 흐름

```
Session 14 (SFT) ─────┐
Session 15 (CPT)  ────┼─→ 16b (본 세션) ─→ before/after 정량 비교
Session 16 (IT)  ─────┘                      LLM Judge로 의미 평가
```

### LLM Judge 활용 패턴 (이미 본 강의에서 사용된 곳)

- **Session 13** Part B 합성 데이터 — 생성 데이터 품질 필터링
- **Session 19b** Rejection Sampling — Best-of-N 선택 기준
- **Session 16b** (지금) — 학습 효과 정량 평가

### 다음 세션 예고

- 🔜 **Session 17 (Part 3 시작)**: LLM 강화학습 개념 — 본 세션의 LLM Judge 점수를 **reward signal로 활용**하는 방법으로 자연스럽게 연결됩니다.

---

### 💡 실습 과제

1. Session 14에서 학습한 SFT 어댑터를 로드해 본 노트북의 평가 함수로 before/after 비교
2. 동일 응답에 대해 BLEU vs BERTScore vs LLM Judge 점수가 얼마나 일치/불일치하는지 분석
3. Pairwise 평가에서 A/B 위치를 바꿔 두 번 실행 → 위치 편향 측정
4. (선택) gpt-4o-mini vs gpt-4o의 Judge 일치율 비교 (비용 vs 품질)

---

In [ ]:
print("✅ Session 16b 완료!")
print("📚 다음 세션: Session 17 — LLM 강화학습 개념 (Part 3 시작)")